# Fine-tune YOLO26n on keys/wallet/sunglasses (Kaggle free GPU)

Companion to `docs/FINE_TUNING_YOLO_GUIDE.md` in the `vision-assist-v1` repo. Run top to bottom on a Kaggle Notebook with **Accelerator = GPU T4 x2** (or P100) enabled (Settings panel, right sidebar).

What this notebook does: downloads LVIS + COCO via Ultralytics' own dataset machinery, runs `prepare_dataset.py` to filter/remap/merge them into the project's 83-class scheme, trains with `freeze=10`, then runs the three-way eval harness against whatever held-out sets you've uploaded as a Kaggle Dataset (see `finetuning/README.md` for how to prepare and attach those).

Nothing in this repo has executed this yet — treat every cell's expected output in comments as what *should* happen, not a guarantee.

In [ ]:
!pip install -q ultralytics==8.4.106
import ultralytics
ultralytics.checks()

## 1. Force Ultralytics to download + convert LVIS and COCO

`lvis.yaml` is ~20.7GB — this is the slow step. We only need Ultralytics' own download/convert machinery to run once; one epoch is the simplest reliable trigger. Interrupt the cell (Kaggle: stop button) right after it logs `Epoch 1/1` starting, or just let the throwaway epoch finish — either way the dataset is now on disk at whatever `ultralytics` settings.yaml points `datasets_dir` to (print it below).

In [ ]:
from ultralytics import YOLO, settings
print("datasets_dir:", settings["datasets_dir"])

model = YOLO("yolo26n.pt")
model.train(data="lvis.yaml", epochs=1, imgsz=640)  # triggers LVIS download+convert; interrupt after epoch 1 starts

In [ ]:
model = YOLO("yolo26n.pt")
model.train(data="coco.yaml", epochs=1, imgsz=640)  # triggers COCO train2017 download+convert; interrupt after epoch 1 starts

## 2. Build the merged 83-class dataset

Upload `finetuning/prepare_dataset.py` from this repo as a Kaggle Dataset (or paste its contents into a cell) before running this. Adjust `--lvis-root`/`--coco-root` to match whatever path cell 1 printed as `datasets_dir`.

In [ ]:
DATASETS_DIR = settings["datasets_dir"]  # from cell 1's print

!python prepare_dataset.py \
    --lvis-root {DATASETS_DIR}/lvis \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/lostfound \
    --coco-sample-size 750

## 3. Train

`freeze=10` — see `docs/FINE_TUNING_YOLO_GUIDE.md` Section 4 for the exact frozen/trainable parameter breakdown (1,115,744 frozen / 1,456,536 trainable, 56.62% of the model trains). A Kaggle session caps at 12 hours; a nano model on a low-thousands-of-images dataset should finish 50-100 epochs on a single T4 well inside that (Section 8 estimates 1-3 hours).

In [ ]:
model = YOLO("yolo26n.pt")
results = model.train(
    data="/kaggle/working/lostfound/lostfound.yaml",
    epochs=100,
    imgsz=640,
    freeze=10,
    optimizer="AdamW",
    lr0=0.001,
    patience=15,
    project="/kaggle/working/runs",
    name="lostfound_yolo26n",
)

## 4. Sanity-check the fine-tuned checkpoint

Quick check that the new classes actually trained — full three-way comparison against the held-out sets lives in `finetuning/evaluate.py`, run separately (see `finetuning/README.md`).

In [ ]:
best = YOLO("/kaggle/working/runs/lostfound_yolo26n/weights/best.pt")
metrics = best.val(data="/kaggle/working/lostfound/lostfound.yaml", split="val")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("per-class mAP50-95 (index 80/81/82 = keys/wallet/sunglasses):", metrics.box.maps[80:83])

## 5. Download the checkpoint

`best.pt` lands under Kaggle's output tab (`/kaggle/working/runs/lostfound_yolo26n/weights/best.pt`) and downloads from there when the notebook finishes. Next step is Section 9 of `FINE_TUNING_YOLO_GUIDE.md`: host it (Hugging Face Hub recommended) and point `FINE_TUNED_YOLO_MODEL_PATH` at it.